In [3]:
import os
import cv2
import numpy as np

# Paths to images and YOLO labels
image_dir = r"C:\project_mrsac\DATASET\train\images"
label_dir = r"C:\project_mrsac\DATASET\train\labels"
mask_dir = r"C:\project_mrsac\DATASET\mask_img"  # Output directory for masks
image_size = (256, 256)  # Resize dimensions

os.makedirs(mask_dir, exist_ok=True)  # Create mask folder if not exists

# Get all image names
image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))]

for image_name in image_files:
    # Read the image to get its size
    image_path = os.path.join(image_dir, image_name)
    img = cv2.imread(image_path)
    h, w, _ = img.shape  # Original image size

    # Create an empty black mask
    mask = np.zeros((h, w), dtype=np.uint8)

    # Find corresponding YOLO annotation file
    label_path = os.path.join(label_dir, image_name.replace(".jpg", ".txt").replace(".png", ".txt"))
    if not os.path.exists(label_path):
        continue  # Skip if no annotation file found

    # Read YOLO annotations
    with open(label_path, "r") as f:
        for line in f:
            data = line.strip().split()
            class_id = int(data[0])  # Extract class (not used, all treated as boundaries)
            points = np.array(data[1:], dtype=float).reshape(-1, 2)  # Convert to numpy array (x, y pairs)

            # Convert normalized YOLO coordinates to pixel coordinates
            polygon = np.array([(int(x * w), int(y * h)) for x, y in points], np.int32)

            # Draw polygon (farm boundary) on the mask
            cv2.polylines(mask, [polygon], isClosed=True, color=255, thickness=2)

    # Resize mask for U-Net
    mask_resized = cv2.resize(mask, image_size)

    # Save the mask
    mask_name = os.path.join(mask_dir, image_name.replace(".jpg", "_mask.png").replace(".png", "_mask.png"))
    cv2.imwrite(mask_name, mask_resized)

print("✅ Mask generation completed! Polygons are now outlined as farm boundaries.")

✅ Mask generation completed! Polygons are now outlined as farm boundaries.
